# M1 — kiểm tra dừng và chạy tiếp ở bước 10 → 20
Dùng GPU T4 và release `source_20260924_072646_069371` để lặp lại phép thử cũ. Mỗi lần chạy ô rehearsal tạo hai run GPU mới. Hai ô cuối chỉ định nghĩa hàm cho pilot độc lập; `Chạy tất cả` không gọi chúng. Đây không phải full M1 hay đánh giá test. Muốn thử mã mới, dùng CLI với release mới; xem `docs/sft_full.md`.

In [ ]:
from pathlib import Path

import torch
from google.colab import drive

assert torch.cuda.is_available(), 'Chọn GPU trong Runtime > Change runtime type'
print('GPU:', torch.cuda.get_device_name(0))
if not Path('/content/drive/MyDrive').is_dir():
    drive.mount('/content/drive')
ROOT = Path('/content/drive/MyDrive/Vietnamese_VQA_NLE')
assert ROOT.is_dir(), f'Sai tài khoản Drive hoặc thiếu thư mục: {ROOT}'
print('Drive:', ROOT)

In [ ]:
import hashlib
import json
import stat
import subprocess
import sys
import zipfile
from tempfile import TemporaryDirectory

SOURCE_RELEASE = 'source_20260924_072646_069371'
EXPECTED_SOURCE_SHA256 = 'a797a32fe2dc5c36196aa7c68ec0bc9b4f52ec222faaa5313c4b49db22d6b333'
SOURCE_DIR = ROOT / 'source/releases'
SOURCE_HASH = SOURCE_DIR / f'{SOURCE_RELEASE}.sha256'
SOURCE_ZIP = SOURCE_DIR / f'{SOURCE_RELEASE}.zip'
if not SOURCE_HASH.is_file() or not SOURCE_ZIP.is_file():
    raise FileNotFoundError(f'Thiếu source release đã ghim: {SOURCE_HASH} hoặc {SOURCE_ZIP}')
checksum_parts = SOURCE_HASH.read_text(encoding='utf-8').split()
if len(checksum_parts) != 2 or checksum_parts != [EXPECTED_SOURCE_SHA256, SOURCE_ZIP.name]:
    raise ValueError(f'Checksum manifest không khớp release đã ghim: {SOURCE_HASH}')
digest = hashlib.sha256()
with SOURCE_ZIP.open('rb') as source:
    for block in iter(lambda: source.read(1024 * 1024), b''):
        digest.update(block)
if digest.hexdigest() != EXPECTED_SOURCE_SHA256:
    raise ValueError(f'Source ZIP checksum mismatch: {SOURCE_ZIP}')

ARCHIVE_ROOT = 'Vietnamese-VQA-Explanation/'
MANIFEST_MEMBER = ARCHIVE_ROOT + 'SOURCE_MANIFEST.json'

def _sha256_stream(source):
    digest = hashlib.sha256()
    for block in iter(lambda: source.read(1024 * 1024), b''):
        digest.update(block)
    return digest.hexdigest()

def _verify_source_archive(archive):
    members = archive.infolist()
    names = [member.filename for member in members]
    if len(names) != len(set(names)) or MANIFEST_MEMBER not in names:
        raise ValueError('Source ZIP has duplicate or missing manifest members')
    manifest_bytes = archive.read(MANIFEST_MEMBER)
    manifest = json.loads(manifest_bytes)
    if not isinstance(manifest, dict) or not manifest:
        raise ValueError('Source manifest must be a nonempty object')
    for relative, expected in manifest.items():
        if (not isinstance(relative, str) or not relative
                or any(part in ('', '.', '..') for part in relative.split('/'))
                or '\\' in relative or ':' in relative):
            raise ValueError(f'Unsafe source path: {relative!r}')
        if (not isinstance(expected, str) or len(expected) != 64
                or any(char not in '0123456789abcdef' for char in expected)):
            raise ValueError(f'Invalid source hash: {relative}')
    expected_names = {ARCHIVE_ROOT + relative for relative in manifest}
    expected_names.add(MANIFEST_MEMBER)
    if set(names) != expected_names or any(
        member.is_dir() or stat.S_IFMT(member.external_attr >> 16) == stat.S_IFLNK
        for member in members
    ):
        raise ValueError('Source ZIP members do not match safe manifest files')
    if archive.testzip() is not None:
        raise ValueError('Source ZIP failed CRC check')
    for relative, expected in manifest.items():
        with archive.open(ARCHIVE_ROOT + relative) as source:
            if _sha256_stream(source) != expected:
                raise ValueError(f'Source file hash mismatch: {relative}')
    return manifest, hashlib.sha256(manifest_bytes).hexdigest()

def _verify_source_tree(repo, manifest, manifest_hash):
    manifest_file = repo / 'SOURCE_MANIFEST.json'
    if not manifest_file.is_file() or manifest_file.is_symlink():
        raise ValueError('Source tree has no regular manifest file')
    with manifest_file.open('rb') as source:
        if _sha256_stream(source) != manifest_hash:
            raise ValueError('Source tree manifest differs from ZIP')
    for relative, expected in manifest.items():
        file = repo / relative
        if not file.is_file() or file.is_symlink():
            raise ValueError(f'Source tree file missing or linked: {relative}')
        with file.open('rb') as source:
            if _sha256_stream(source) != expected:
                raise ValueError(f'Source tree file hash mismatch: {relative}')
    executable = set()
    for folder in ('src', 'scripts', 'configs'):
        for file in (repo / folder).rglob('*'):
            relative = file.relative_to(repo)
            if (file.suffix in ('.py', '.yaml', '.yml')
                    and not any(part == '__pycache__' or part.endswith('.egg-info')
                                for part in relative.parts)):
                executable.add(relative.as_posix())
    executable.update(file.name for file in repo.glob('*.py'))
    unexpected = executable - manifest.keys()
    if unexpected:
        raise ValueError(f'Unexpected executable source files: {sorted(unexpected)}')
WORK_SOURCE = Path('/content/vqa_sources') / SOURCE_HASH.stem
REPO = WORK_SOURCE / 'Vietnamese-VQA-Explanation'
with zipfile.ZipFile(SOURCE_ZIP) as archive:
    manifest, manifest_hash = _verify_source_archive(archive)
    if REPO.exists():
        _verify_source_tree(REPO, manifest, manifest_hash)
    else:
        if WORK_SOURCE.exists():
            raise FileExistsError(f'Incomplete source tree; use a clean runtime: {WORK_SOURCE}')
        WORK_SOURCE.parent.mkdir(parents=True, exist_ok=True)
        with TemporaryDirectory(prefix='source-stage-', dir=WORK_SOURCE.parent) as stage:
            archive.extractall(stage)
            _verify_source_tree(Path(stage) / 'Vietnamese-VQA-Explanation',
                                manifest, manifest_hash)
            Path(stage).rename(WORK_SOURCE)
        _verify_source_tree(REPO, manifest, manifest_hash)
assert (REPO / 'scripts/train_sft_full.py').is_file(), 'Release thiếu runner M1'
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', '-e', f'{REPO}[train]'],
    check=True,
)
print('Source verified:', SOURCE_RELEASE, EXPECTED_SOURCE_SHA256)

In [ ]:
BUNDLE = ROOT / 'datasets/releases/vivqax_image_disjoint_v1'
EXPECTED_BUNDLE_SHA256 = '46846a81c7b3594d4fc9670ec51019cfd652f1eb169af7978e39dc5a40a55522'
WORK_DATA = Path('/content/vqa_work/vivqax_image_disjoint_v1')
assert hashlib.sha256((BUNDLE / 'bundle.json').read_bytes()).hexdigest() == EXPECTED_BUNDLE_SHA256
bundle = json.loads((BUNDLE / 'bundle.json').read_text(encoding='utf-8'))
command = [sys.executable, str(REPO / 'scripts/verify_coco_bundle.py'), str(BUNDLE)]
if not WORK_DATA.exists():
    command += ['--extract-to', str(WORK_DATA)]
subprocess.run(command, cwd=REPO, check=True)
assert len(list((WORK_DATA / 'images').glob('*.jpg'))) == bundle['total_images']
checksums = WORK_DATA / 'metadata/full_image_checksums.csv'
assert hashlib.sha256(checksums.read_bytes()).hexdigest() == bundle['image_checksums_sha256']
assert (WORK_DATA / 'metadata/all.jsonl').is_file()
print('Data verified:', bundle['total_images'], 'images')

## Thử dừng ở bước 10, chạy tiếp đến bước 20
Chạy sau ba ô mount/source/data. Script tạo run mới và chỉ ghi `resume_rehearsal.json` khi checkpoint, nguồn dữ liệu và mã, loss và trọng số LoRA-B qua kiểm tra. Run cũ được giữ nguyên.

In [ ]:
RUN_ROOT = ROOT / 'runs'
ENV = 'configs/environments/colab_full.example.yaml'
report_pattern = 'sft-optimizer-pilot_*/resume_rehearsal.json'
before_reports = {path.resolve() for path in RUN_ROOT.glob(report_pattern)}
subprocess.run(
    [sys.executable, '-u', str(REPO / 'scripts/rehearse_m1_resume.py'),
     '--environment', ENV], cwd=REPO, check=True
)
new_reports = [path.resolve() for path in RUN_ROOT.glob(report_pattern)
               if path.resolve() not in before_reports]
assert len(new_reports) == 1, f'Không xác định được report rehearsal mới: {new_reports}'
REHEARSAL_REPORT = json.loads(new_reports[0].read_text(encoding='utf-8'))
assert REHEARSAL_REPORT['status'] == 'passed'
assert REHEARSAL_REPORT['max_lora_b_weight_change'] > 0
print('Rehearsal report:', new_reports[0])
print('LoRA-B max change:', REHEARSAL_REPORT['max_lora_b_weight_change'])
print('Đây là kiểm tra resume kỹ thuật, chưa phải M1 đầy đủ hoặc điểm chất lượng.')

## Pilot độc lập (tùy chọn)
Hai ô sau chỉ định nghĩa hàm. Nếu cần chạy lại pilot, gọi `pilot_run, pilot_adapter = run_standalone_pilot()` trong ô mới; sau đó có thể gọi `reload_standalone_pilot_adapter(pilot_run, pilot_adapter)`. Không gọi hai hàm này trong lượt thử 10 → 20.

In [ ]:
def run_standalone_pilot():
    import math

    run_root = ROOT / 'runs'
    environment = 'configs/environments/colab_full.example.yaml'
    subprocess.run(
        [sys.executable, '-u', str(REPO / 'scripts/run_pipeline.py'),
         'sft-full-preflight', '--environment', environment], cwd=REPO, check=True
    )
    before = {path.name for path in run_root.glob('sft-optimizer-pilot_*')}
    subprocess.run(
        [sys.executable, '-u', str(REPO / 'scripts/run_pipeline.py'),
         'sft-optimizer-pilot', '--environment', environment], cwd=REPO, check=True
    )
    created = [path for path in run_root.glob('sft-optimizer-pilot_*') if path.name not in before]
    assert len(created) == 1, f'Không xác định được run pilot mới: {created}'
    run = created[0]
    execution = json.loads((run / 'execution.json').read_text(encoding='utf-8'))
    report = json.loads((run / 'training/training_result.json').read_text(encoding='utf-8'))
    assert execution['status'] == 'passed' and execution['returncode'] == 0
    assert report['status'] == 'passed' and report['experiment_id'] == 'M1_SFT_PILOT'
    assert report['global_step'] == 20 and report['max_lora_weight_change'] > 0
    assert math.isfinite(report['final_validation']['eval_loss'])
    assert report['train_seconds'] > 0 and report['peak_vram_gib'] > 0
    checkpoint = Path(report['best_model_checkpoint'])
    assert all((checkpoint / name).is_file() for name in (
        'trainer_state.json', 'optimizer.pt', 'scheduler.pt'
    ))
    adapter = Path(report['adapter'])
    assert (adapter / 'adapter_config.json').is_file()
    assert any((adapter / name).is_file() for name in (
        'adapter_model.safetensors', 'adapter_model.bin'
    ))
    print('PILOT RUN:', run)
    for key in (
        'global_step', 'train_seconds', 'peak_vram_gib',
        'max_lora_weight_change', 'best_eval_loss'
    ):
        print(key, report[key])
    steps_per_epoch = math.ceil(29459 / 8)
    hours = round(report['train_seconds'] * steps_per_epoch / 20 / 3600, 1)
    print('Rough full-epoch hours:', hours)
    print('Ước lượng rất thô, gồm overhead eval/checkpoint; chưa tự chạy full M1 hoặc test.')
    return run, adapter

In [ ]:
def reload_standalone_pilot_adapter(run, adapter):
    import yaml

    from vqa_explanation.data.io import read_jsonl
    from vqa_explanation.inference.qwen25_vl import Qwen25VLRunner

    validation = next(
        row for row in read_jsonl(WORK_DATA / 'metadata/all.jsonl')
        if row['split'] == 'validation'
    )
    model_config = REPO / 'configs/model/qwen25_vl_3b.yaml'
    model = yaml.safe_load(model_config.read_text(encoding='utf-8'))['model']
    reloaded = Qwen25VLRunner(
        model['id'], model['revision'], load_in_4bit=True,
        min_pixels=200704, max_pixels=200704, adapter_path=str(adapter)
    )
    image = WORK_DATA / validation['image_path']
    assert image.is_file()
    prediction = reloaded.generate(image, validation['question'], max_new_tokens=128)
    assert prediction.strip(), 'Adapter reload returned an empty prediction'
    reload_report = {
        'status': 'passed', 'experiment_id': 'M1_SFT_PILOT',
        'question_id': validation['question_id'], 'prediction': prediction,
        'note': 'Load/generation smoke only; not quality evaluation',
    }
    with (run / 'adapter_reload.json').open('x', encoding='utf-8') as output:
        json.dump(reload_report, output, ensure_ascii=False, indent=2)
    print('Adapter reload passed on validation question:', validation['question_id'])
    print('Đây chỉ là kiểm tra nạp/suy luận, không phải điểm chất lượng.')
    return reload_report